## Self-Supervised Speech Recognition - Toronto Dataset

In [1]:
%load_ext autoreload
%autoreload 2


In [11]:
from pathlib import Path
import pandas as pd
import torch
from src.dataset import build_timit_df, map_phonemes, DataCollatorCTCWithPadding
import os
import json
import lightning as L
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint
from transformers import (
  Wav2Vec2PhonemeCTCTokenizer,
  Wav2Vec2FeatureExtractor,
  Wav2Vec2Processor,
  Data2VecAudioForCTC,
)
import evaluate
from sklearn.model_selection import GroupKFold

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
DATA_DIR = Path("data")
TIMIT = DATA_DIR / Path("timit")
TIMIT_SR = 16000

PHONE_MAP = {
    "aa": "aa", "ae": "ae", "ah": "ah", "ao": "aa", "aw": "aw",
    "ax": "ah", "ax-h": "ah", "axr": "er", "ay": "ay",
    "b": "b", "bcl": "sil",
    "ch": "ch",
    "d": "d", "dcl": "sil", "dh": "dh", "dx": "dx",
    "eh": "eh", "el": "l", "em": "m", "en": "n", "eng": "ng",
    "epi": "sil", "er": "er", "ey": "ey",
    "f": "f",
    "g": "g", "gcl": "sil",
    "h#": "sil", "hh": "hh", "hv": "hh",
    "ih": "ih", "ix": "ih", "iy": "iy",
    "jh": "jh",
    "k": "k", "kcl": "sil",
    "l": "l",
    "m": "m",
    "n": "n", "ng": "ng", "nx": "n",
    "ow": "ow", "oy": "oy",
    "p": "p", "pau": "sil", "pcl": "sil",
    "q": None,
    "r": "r",
    "s": "s", "sh": "sh",
    "t": "t", "tcl": "sil", "th": "th",
    "uh": "uh", "uw": "uw", "ux": "uw",
    "v": "v",
    "w": "w",
    "y": "y",
    "z": "z", "zh": "sh",
}

assert len(PHONE_MAP) == 61

TIMIT_META_TEST = TIMIT / "test_data.csv"
TIMIT_META_TRAIN = TIMIT / "train_data.csv"

TIMIT_TEST = TIMIT / "data/test"
TIMIT_TRAIN = TIMIT / "data/train"

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"


In [3]:
PARQUET_TEST = TIMIT / "test.parquet"
PARQUET_TRAIN = TIMIT / "train.parquet"

if not PARQUET_TEST.exists():
  timit_test_meta  = pd.read_csv(TIMIT_META_TEST)
  timit_test_df  = build_timit_df(timit_test_meta, TIMIT / "data", TIMIT_SR)
  timit_test_df["phonemes_39"] = timit_test_df["phonemes"].apply(lambda x: map_phonemes(x, PHONE_MAP))
  timit_test_df.to_parquet(PARQUET_TEST)
else:
  timit_test_df = pd.read_parquet(PARQUET_TEST)

if not PARQUET_TRAIN.exists():
  timit_train_meta = pd.read_csv(TIMIT_META_TRAIN)
  timit_train_df = build_timit_df(timit_train_meta, TIMIT / "data", TIMIT_SR)
  timit_train_df["phonemes_39"] = timit_train_df["phonemes"].apply(lambda x: map_phonemes(x, PHONE_MAP))
  timit_train_df.to_parquet(PARQUET_TRAIN)
else:
  timit_train_df = pd.read_parquet(PARQUET_TRAIN)


In [4]:
VOCAB = TIMIT / "vocab.json"

vocab_dict = {v: i for i, v in enumerate(set(PHONE_MAP.values())) if v is not None}
vocab_dict["<unk>"] = len(vocab_dict)
vocab_dict["<pad>"] = len(vocab_dict)

with open(VOCAB, "w", encoding="utf-8") as f:
  json.dump(vocab_dict, f, ensure_ascii=False)


In [6]:
class Data2VecCTC(L.LightningModule):
  def __init__(self, model, processor):
    super().__init__()
    self.model = model
    self.processor = processor
    self.per = evaluate.load("per")

  def on_train_epoch_start(self):
    self.model.train()

  def training_step(self, batch, _):
    loss = self.model(**batch).loss
    self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def validation_step(self, batch, _):
    out = self.model(**batch)
    self.log("val_loss", out.loss, prog_bar=True)

    pred_ids = torch.argmax(out.logits, dim=-1)

    labels = batch["labels"].clone()
    labels[labels == -100] = self.processor.tokenizer.pad_token_id

    preds = self.processor.batch_decode(pred_ids)
    refs = self.processor.batch_decode(labels, group_tokens=False)

    self.per.add_batch(predictions=preds, references=refs)

  def on_validation_epoch_end(self):
    self.log("val_per", self.per.compute(), prog_bar=True)

  def configure_optimizers(self):
    return torch.optim.AdamW(self.model.parameters(), lr=1e-4)


In [ ]:
collator = DataCollatorCTCWithPadding(processor=processor)

gkf = GroupKFold(n_splits=5)
train_idx, val_idx = next(gkf.split(toronto_train_df, groups=toronto_train_df["speaker_id"]))

toronto_tr_df  = toronto_train_df.iloc[train_idx]
toronto_val_df = toronto_train_df.iloc[val_idx]

toronto_train_ds = TorontoDataset(toronto_tr_df, processor, TORONTO, features_column="input_values")
toronto_val_ds = TorontoDataset(toronto_val_df, processor, TORONTO, features_column="input_values")
toronto_test_ds = TorontoDataset(toronto_test_df, processor, TORONTO, features_column="input_values")

toronto_train_dl = DataLoader(
  toronto_train_ds,
  batch_size=8,
  shuffle=True,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
  persistent_workers=True,
)

toronto_val_dl = DataLoader(
  toronto_val_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

toronto_test_dl = DataLoader(
  toronto_test_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

logger = CSVLogger("logs", name="data2vec_finetune_val")
m = Data2VecCTC(model, processor)

checkpoint = ModelCheckpoint(
  monitor="val_wer",
  mode="min",
  save_top_k=1,
  filename="{epoch}-{val_wer:.3f}",
)

trainer = L.Trainer(
  accelerator=device,
  logger=logger,
  max_epochs=5,
  precision="bf16-mixed",
  callbacks=[checkpoint],
)

model.train()

for param in model.data2vec_audio.parameters():
    param.requires_grad = False

trainer.fit(m, train_dataloaders=toronto_train_dl, val_dataloaders=toronto_val_dl)
trainer.validate(m, dataloaders=toronto_test_dl, ckpt_path="best")


In [ ]:
BACKBONE_MODEL = "facebook/data2vec-audio-base"

tokenizer = Wav2Vec2PhonemeCTCTokenizer(VOCAB)

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(BACKBONE_MODEL)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

model = Data2VecAudioForCTC.from_pretrained(
  BACKBONE_MODEL,
  vocab_size=len(vocab_dict),
  pad_token_id=tokenizer.pad_token_id,
  ctc_zero_infinity=True,
  ctc_loss_reduction="mean",
  attn_implementation="sdpa",
)

model.freeze_feature_encoder()
